In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys 
sys.path.append('../src')

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "5"

In [3]:
import torch
import shutil
from mapanything.models import MapAnything
from mapanything.utils.image import load_images
from huggingface_hub import snapshot_download
from rescue.utils import sample_video
from rescue import ges_utils
from mapanything.utils.hf_utils.viz import predictions_to_glb
from mapanything.utils.geometry import depthmap_to_world_frame
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
save_path = snapshot_download("facebook/map-anything", repo_type="model", local_dir="../generated/map-anything")
model = MapAnything.from_pretrained(save_path).to(device)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading pretrained dinov2_vitg14 from torch hub


Using cache found in /home/srohatgi/.cache/torch/hub/facebookresearch_dinov2_main


Loading weights from local directory


In [8]:
temp_dir_path = '../generated/temp'
if os.path.isdir(temp_dir_path):
    shutil.rmtree(temp_dir_path)

os.makedirs(temp_dir_path, exist_ok=True)
files_dir, sampled_indices = sample_video('../generated/disaster_city_lawnmow_loop.mp4', fps=1, output_dir =temp_dir_path)

In [9]:
len(sampled_indices)

67

In [10]:
num_views = -1

# views = load_images('../generated/sampled_disaster_city/')[:num_views]
views = load_images(files_dir)[:num_views]

poses, c2w_list, K_list, orig_width, orig_height = ges_utils.convert_ges_to_mapanything_from_file('../generated/disaster_city_lawnmow_loop.json', ref_frame = 0)
poses = [poses[i] for i in sampled_indices[:num_views]]
c2w_list = [c2w_list[i] for i in sampled_indices[:num_views]]
K_list = [K_list[i] for i in sampled_indices[:num_views]]

assert len(views) == len(poses) == len(c2w_list) == len(K_list), 'Lengths of views, poses, c2w_list, and K_list must be equal'

for i in range(len(views)):
    h_new, w_new = views[i]['true_shape'][0]   
    K = K_list[i].copy()
    scale_x = w_new / orig_width            
    scale_y = h_new / orig_height           
    K[0, 0] *= scale_x   # fl_x
    K[0, 2] *= scale_x   # cx
    K[1, 1] *= scale_y   # fl_y
    K[1, 2] *= scale_y   # cy

    device = views[i]['img'].device
    views[i]['intrinsics']    = torch.tensor(K, dtype=torch.float32, device=device)[None]
    views[i]['camera_poses']  = torch.tensor(c2w_list[i], dtype=torch.float32, device=device)[None]
    views[i]['is_metric_scale'] = False

In [11]:
len(views)

66

In [12]:
import numpy as np

# Take frame 0 - its ENU position should be exactly [0,0,0]
# and its forward vector (col 2 of R) should point roughly downward (nadir view)
c2w_0 = c2w_list[0]   # before gl2cv
print("Position (should be ~[0,0,0]):", c2w_0[:3, 3])

# Camera axes in ENU world
right   = c2w_0[:3, 0]   # should point East-ish
up_cam  = c2w_0[:3, 1]   # in OpenCV: should point DOWN in world
forward = c2w_0[:3, 2]   # in OpenCV: should point INTO scene (roughly toward ground)

print("Forward vector (col 2):", np.round(forward, 3))
print("  Z component:", forward[2], "← negative means pointing downward (correct for nadir)")


Position (should be ~[0,0,0]): [0. 0. 0.]
Forward vector (col 2): [ 0.  0. -1.]
  Z component: -1.0 ← negative means pointing downward (correct for nadir)


In [13]:
# views[0].keys()

In [14]:
# # Verify: view i's idx field should match i
# assert all(views[i]['idx'] == i for i in range(len(views))), "load_images reordered files"

# # Verify: sampled_indices should be strictly increasing (no frame reordering)
# assert all(sampled_indices[i] < sampled_indices[i+1] for i in range(len(sampled_indices)-1)), \
#     "sampled_indices not monotonically increasing"

# print(f"First 5 sampled source frames: {sampled_indices[:5]}")
# print(f"views[0] true_shape: {views[0]['true_shape']}")

In [15]:
predictions = model.infer(
    views,                            # Input views
    memory_efficient_inference=True,  # Trades off speed for more views (up to 2000 views on 140 GB). Trade off is negligible - see profiling section
    minibatch_size=None,              # Minibatch size for memory-efficient inference (use 1 for smallest GPU memory consumption). Default is dynamic computation based on available GPU memory.
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=True,                  # Apply masking to dense geometry outputs
    mask_edges=True,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
    use_multiview_confidence=True,   # Enable multi-view depth consistency based confidence in place of learning-based one
)

Checking triangle intersections: 100%|██████████| 2/2 [00:00<00:00, 24.66it/s]


In [ ]:
# Build the predictions dict that predictions_to_glb expects
world_points_list = []
images_list       = []
extrinsic_list    = []
mask_list         = []

for pred in predictions:
    pts3d, valid_mask = depthmap_to_world_frame(
        pred["depth_z"][0].squeeze(-1),
        pred["intrinsics"][0],
        pred["camera_poses"][0],
    )

    mask = pred["mask"][0].squeeze(-1).cpu().numpy().astype(bool)
    mask = mask & valid_mask.cpu().numpy()

    world_points_list.append(pts3d.cpu().numpy())
    images_list.append(pred["img_no_norm"][0].cpu().numpy())
    extrinsic_list.append(pred["camera_poses"][0].cpu().numpy())   # (4, 4) cam2world
    mask_list.append(mask)

pred_dict = {
    "world_points": np.stack(world_points_list),  # (S, H, W, 3)
    "images":       np.stack(images_list),         # (S, H, W, 3)
    "extrinsic":    np.stack(extrinsic_list),      # (S, 3, 4)
    "final_mask":   np.stack(mask_list),           # (S, H, W)
}

# Export to GLB
glbscene = predictions_to_glb(
    pred_dict,
    show_cam=True,
    as_mesh=False,       # True = mesh, False = point cloud
    conf_percentile=0,   # 0 = keep all points
)
glbscene.export("../generated/reconstruction.glb")

# Additional step: load the exported GLB, convert/copy it to Blender specs, and save as a separate GLB for Blender



Building GLB scene
Using Pointmap Branch
GLB Scene built


In [ ]:
# Convert the exported GLB to Blender's coordinate system (Z-up).
import trimesh
import numpy as np

input_glb_path = "../generated/reconstruction.glb"
output_blender_glb_path = "../generated/reconstruction_blender.glb"

# Load the mesh or scene
mesh_or_scene = trimesh.load(input_glb_path)

# Rotate -90 degrees (not +90!) around X-axis to convert Y-up (GLB default) to Z-up (Blender)
# A positive 90 deg will flip Z-down instead of Z-up (Blender expects Z-up as positive)
transform = trimesh.transformations.rotation_matrix(np.radians(-90), [1, 0, 0])

if isinstance(mesh_or_scene, trimesh.Scene):
    # Apply transform to all geometries in the scene
    for geom in mesh_or_scene.geometry.values():
        geom.apply_transform(transform)
    mesh_or_scene.export(output_blender_glb_path)
else:
    mesh_or_scene.apply_transform(transform)
    mesh_or_scene.export(output_blender_glb_path)

print(f"Saved Z-up GLB (for Blender) to {output_blender_glb_path}")

Saved Z-up GLB (for Blender) to ../generated/reconstruction_blender.glb
